In [ ]:
// ==== Setup ====

import java.util.*;

interface MyListDouble {
    // appends e at the end
    void add(double e);

    // inserts e at pos, elements pos..size-1 move up by one
    // valid: 0 <= pos <= size
    void add(int pos, double e);

    // removes and returns the element at pos, elements pos+1..size-1 move down by one
    // valid: 0 <= pos < size
    double remove(int pos);

    // valid: 0 <= pos < size
    double get(int pos);

    // replaces the element at pos, returns the old one
    // valid: 0 <= pos < size
    double set(int pos, double e);

    // position of the first occurrence of e, or -1
    int indexOf(double e);

    int size();
    boolean isEmpty();
    void clear();
}

# Linked lists

## Where we are

Array-based list:

| | front | middle | back |
|---|---|---|---|
| `add` | `O(N)` | `O(N)` | `O(1)` amortized |
| `remove` | `O(N)` | `O(N)` | `O(1)` |
| `get`, `set` | `O(1)` | `O(1)` | `O(1)` |

- The cost at the front comes from **shifting**: the elements must occupy consecutive cells.
- Idea: drop that requirement. Each element stores **where the next one is**.

## Nodes

- **Node**: one element + a reference to the next node.
- **Linked list**: a chain of nodes. We keep a reference to the first one, `head`.
- The last node has `next == null`.

In [ ]:
class Node {
    double data;
    Node next;          // null: no next node

    Node(double data) { this.data = data; }
}

In [ ]:
Node a = new Node(3.5);
Node b = new Node(1.0);
Node c = new Node(7.25);
a.next = b;
b.next = c;

Node head = a;
System.out.println(head.data + " " + head.next.data + " " + head.next.next.data);
System.out.println(head.next.next.next);

- How many objects did we create? How many references point to the node holding `1.0`?
- What does `head.next.next.next.data` do?
- The nodes are anywhere in memory, not in consecutive cells.

### Generic nodes

```java
class Node<E> {
    E data;
    Node<E> next;
}
```

`Node<String>`, `Node<Integer>`, ... Same structure for any element type. We keep `double`, as for `MyArrayListDouble`.

## Traversal

Start at `head`, follow `next` until `null`.

```java
Node current = head;
while (current != null) {
    // use current.data
    current = current.next;
}
```

In [ ]:
void printAll(Node head) {
    Node current = head;
    while (current != null) {
        System.out.print(current.data + " ");
        current = current.next;
    }
    System.out.println();
}

int length(Node head) {
    int count = 0;
    for (Node current = head; current != null; current = current.next)
        count++;
    return count;
}

printAll(head);
System.out.println(length(head) + " " + length(null));

### Stopping at the last node

Condition: `current.next != null` instead of `current != null`.

In [ ]:
// the last node, or null if the list is empty
Node last(Node head) {
    if (head == null) return null;
    Node current = head;
    while (current.next != null)
        current = current.next;
    return current;
}

System.out.println(last(head).data);

Why the `if` before the loop?

### Moving `head`

Inside a list class, where `head` is a **field**:

```java
void printList() {
    while (head != null) {
        System.out.print(head.data + " ");
        head = head.next;
    }
}
```

- Does it print the list?
- What is wrong with it?
- In `printAll(Node head)` above, what if we moved the parameter `head` instead of `current`?

# `MyLinkedListDouble`

## Fields

- `head`: first node, `null` if the list is empty
- `Node`: a `private static` class inside the list. Users of the list never see nodes.

```java
class MyLinkedListDouble implements MyListDouble {
    private static class Node {
        double data;
        Node next;

        Node(double data) { this.data = data; }
    }

    private Node head;
    ...
}
```

We will need **two more fields**: we add them when they come up.

## Adding at the front

1. create the node
2. link it to the current first node
3. move `head`

```java
public void addFirst(double e) {
    Node n = new Node(e);
    n.next = head;
    head = n;
}
```

- Swap steps 2 and 3: what happens?
- Does it work on an empty list?
- `O(1)`

## Adding at the back

Walk to the last node, link the new one after it.

```java
public void add(double e) {
    Node n = new Node(e);
    if (head == null) {             // the list was empty
        head = n;
        return;
    }
    Node current = head;
    while (current.next != null)
        current = current.next;
    current.next = n;
}
```

- Cost of one `add`? Of `N` appends to an empty list?
- How can we avoid the walk?

### First new field: `tail`

```java
private Node tail;      // last node, null if empty
```

```java
public void add(double e) {
    Node n = new Node(e);
    if (tail == null) head = n;     // the list was empty
    else tail.next = n;
    tail = n;
}
```

- `O(1)`
- Every method that can change the last node now has to update `tail`. Which line does `addFirst` need?

```java
public void addFirst(double e) {
    Node n = new Node(e);
    n.next = head;
    head = n;
    if (tail == null) tail = n;     // the list was empty
}
```

## Reaching a position

```java
// the node at pos, assumes 0 <= pos < size
private Node nodeAt(int pos) {
    Node current = head;
    for (int i = 0; i < pos; i++)
        current = current.next;
    return current;
}

public double get(int pos) {
    checkIndex(pos);
    return nodeAt(pos).data;
}
```

- `get`, `set`: `pos` steps, `O(N)`
- Array: `data[pos]` is `O(1)`. Where does the difference come from?
- `checkIndex` needs the number of elements. How do we get it?

### Second new field: `size`

- Counting the nodes (`length` above): `O(N)` at every call
- Keep the count instead:

```java
private int size;       // number of nodes
```

- `size()`, `isEmpty()`, `checkIndex`: `O(1)`
- every add: `size++`, every remove: `size--`

**Invariants**, true before and after every method:
- empty list: `head == null`, `tail == null`, `size == 0`
- non-empty list: walking from `head`, we reach `tail` after `size - 1` steps, and `tail.next == null`

## Searching

```java
public int indexOf(double e) {
    int pos = 0;
    for (Node current = head; current != null; current = current.next) {
        if (current.data == e) return pos;
        pos++;
    }
    return -1;
}
```

`O(N)`, as for the array.

## Inserting at a position

- Inserting at `pos`: which node's `next` has to change?
- Walk to the node **before** `pos`, then change two links.

```java
public void add(int pos, double e) {
    if (pos < 0 || pos > size)                  // pos == size is allowed
        throw new IndexOutOfBoundsException("pos " + pos + ", size " + size);
    if (pos == 0)    { addFirst(e); return; }
    if (pos == size) { add(e);      return; }
    Node before = nodeAt(pos - 1);
    Node n = new Node(e);
    n.next = before.next;
    before.next = n;
    size++;
}
```

- Swap the two link assignments: what happens?
- Why is `pos == size` a separate case? The general case would link the node correctly.
- Cost: walking `O(pos)` + linking `O(1)`. Nothing is shifted.

## Removing from the front

```java
public double removeFirst() {
    if (head == null) throw new NoSuchElementException("empty list");
    double removed = head.data;
    head = head.next;
    if (head == null) tail = null;  // removed the only node
    size--;
    return removed;
}
```

- The old first node: no reference to it is left. The **garbage collector** reclaims its memory.
- Without the `tail = null` line: which invariant breaks? What goes wrong at the next `add`?
- `O(1)`

## Removing at a position

- Again: walk to the node **before**.
- Bypass the removed node: `before.next = before.next.next`

```java
public double remove(int pos) {
    checkIndex(pos);
    if (pos == 0) return removeFirst();
    Node before = nodeAt(pos - 1);
    double removed = before.next.data;
    before.next = before.next.next;
    if (before.next == null) tail = before;     // removed the last node
    size--;
    return removed;
}
```

### Removing the last node

- `tail` points to the last node, but we need the one **before** it.
- `remove(size - 1)` walks the whole list: `O(N)`, even with `tail`.

## Clearing

```java
public void clear() {
    head = null;
    tail = null;
    size = 0;
}
```

`O(1)`: every node becomes unreachable.